In [ ]:
import torch
import torch.nn.functional as F
from torchvision import transforms
from PIL import Image, ImageDraw


from src.cnn.VisDroneCNN import VisDroneCNN
from src.dataset.Dataset import VisDrone
from src.dataset.collate import collate_fn

In [ ]:
transf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((1024, 1024))
])

### Build model
Wczytanie modelu oraz dokonanie predykcji dla wybranych obrazów.

In [ ]:
train_dev_data = "data/VisDrone_Dataset/VisDrone2019-DET-test-dev/images"
train_labels = "data/VisDrone_Dataset/VisDrone2019-DET-test-dev/labels"

visdrone_test = VisDrone(data_dir=train_dev_data, labels_dir=train_labels, transform=transf)
dataset = torch.utils.data.DataLoader(visdrone_test,
                                      batch_size=8,
                                      shuffle=True,
                                      num_workers=4,
                                      collate_fn=collate_fn,
                                      pin_memory=True,
                                      prefetch_factor=2)


cnn = VisDroneCNN(S=16, B_boxes=1, C=10)
cnn.load_state_dict(torch.load('first.pth'))
device = 'cuda' if torch.cuda.is_available() else 'cpu'

images, labels = next(iter(dataset))

#cnn.to(device)
#images.to(device)
with torch.no_grad():
    output: list = cnn(images)

In [ ]:
prediction = output[0]

### Additional Transforms
Dodatkowe transformacje danych wyjściowych. Niezbędne ponieważ model działa na danych nie poddanych na wyjściowej warstwie funkcją takim jak softmax lub sigmoid (jest to robione w trakcie treningu w funkcjach straty).

In [ ]:
def decode_prediction(prediction: torch.Tensor) -> tuple:
    bx = torch.sigmoid(prediction[..., 0])
    by = torch.sigmoid(prediction[..., 1])
    bw = torch.sigmoid(prediction[..., 2])
    bh = torch.sigmoid(prediction[..., 3])
    obj = torch.sigmoid(prediction[..., 4])
    cls = F.softmax(prediction[..., 5:], dim=-1)
    return bx, by, bw, bh, obj, cls

In [ ]:
cnn.eval()
with torch.no_grad():
    bx, by, bw, bh, obj, cls = decode_prediction(prediction = output)
    print("obj min/max:", obj.min().item(), obj.max().item())

    print("cls suma w jednej komórce:", cls[0, 0, 0].sum().item())

    print("bx min/max:", bx.min().item(), bx.max().item())

In [ ]:
def yolo_to_pixels(bx, by, bw, bh, img_w: int = 256, img_h: int = 256) -> torch.Tensor:
    #TODO: This function should return Yolo type (Used in dataset) for consistency.
    S = bx.shape[-1]
    cell_w = img_w / S
    cell_h = img_h / S

    offset_x = torch.arange(S).float().view(1, 1, S).expand(bx.shape[0], S, S)
    offset_y = torch.arange(S).float().view(1, S, 1).expand(by.shape[0], S, S)

    cx = (offset_x + bx) * cell_w
    cy = (offset_y + by) * cell_h

    bw_px = bw * img_w
    bh_px = bh * img_h

    x1 = cx - bw_px / 2
    x2 = cx + bw_px / 2
    y1 = cy - bh_px / 2
    y2 = cy + bh_px / 2

    boxes = torch.stack([x1, y1, x2, y2], dim=-1)

    return boxes

In [ ]:
boxes = yolo_to_pixels(bx, by, bw, bh, img_w=256, img_h=256)

print("boxes.shape:", boxes.shape)          # [B, S, S, 4]
print("x1 min/max:", boxes[..., 0].min().item(), boxes[..., 0].max().item())
print("y1 min/max:", boxes[..., 1].min().item(), boxes[..., 1].max().item())
print("x2 min/max:", boxes[..., 2].min().item(), boxes[..., 2].max().item())
print("y2 min/max:", boxes[..., 3].min().item(), boxes[..., 3].max().item())

### Filtering
Należy odfiltrować garbage 

In [ ]:
def filter_predictions(boxes, obj, cls, conf_threshold: float = 0.5):
    B = boxes.shape[0]
    results = []
    for b in range(B):
        class_scores, class_ids = cls[b].max(dim=-1)
        scores = obj[b] * class_scores

        mask = scores > conf_threshold

        results.append({
            "boxes": boxes[b][mask],
            "scores": scores[mask],
            "class_ids": class_ids[mask]
        })
    return results

In [ ]:
results = filter_predictions(boxes, obj, cls, conf_threshold=0.55)

for b, r in enumerate(results):
    print(f"Obraz {b}: {len(r['boxes'])} boksów po filtrowaniu")

In [ ]:
CLASS_NAMES = [
    "pedestrian", "people", "bicycle", "car", "van",
    "truck", "tricycle", "awning-tricycle", "bus", "motor"
]

COLORS = [
    (255, 56,  56),  (255, 157, 151), (255, 112,  31),
    (255, 178,  29), (207, 210,  49), (72,  249,  10),
    (146, 204,  23), (61,  219, 134), (26,  147,  52),
    (0,  212, 187),
]

def draw_predictions(image: Image.Image, result: dict):
    draw = ImageDraw.Draw(image)

    for box, score, cls_id in zip(result["boxes"], result["scores"], result["class_ids"]):
        x1, y1, x2, y2 = box.tolist()
        cls_id = int(cls_id)
        color  = COLORS[cls_id]
        label  = f"{CLASS_NAMES[cls_id]}: {score:.2f}"

        draw.rectangle([x1, y1, x2, y2], outline=color, width=2)

        text_bbox = draw.textbbox((x1, y1), label)
        draw.rectangle(text_bbox, fill=color)
        draw.text((x1, y1), label, fill=(255, 255, 255))

    return image

In [ ]:
img_np = images[0].permute(1, 2, 0).cpu().numpy()
img_np = (img_np * 255).clip(0, 255).astype("uint8")
pil_img = Image.fromarray(img_np)

pil_img = draw_predictions(pil_img, results[0])
pil_img.show()